# ResNet18: Hyperparameter Tuning + Full Data-Fraction Sweep
Continues from `food101_setup.ipynb`. Assumes `train_10`, `train_50`, `train_100`, `test_set`, `PROJECT_DIR`, `SEED`, `device` already exist in this session.



## 0. Setup

In [ ]:
%pip install -Uq "mlop[full]"
import mlop
import torch
import torch.nn as nn
import numpy as np
import random
import os
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader, random_split
from tqdm.auto import tqdm

NUM_CLASSES = 101
BATCH_SIZE = 32

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# If mlop.ai dashboard is erroring, this call may still succeed and queue/sync later.
# If it hard-fails, comment out mlop calls below temporarily and log to a local list instead
# (see the LOCAL_FALLBACK note in the tuning loop).
mlop.login()

## 1. Shared train/eval/loader helpers

In [ ]:
def make_train_val_loaders(dataset, val_frac=0.10, batch_size=BATCH_SIZE, seed=SEED):
    n_val = int(len(dataset) * val_frac)
    n_train = len(dataset) - n_val
    generator = torch.Generator().manual_seed(seed)
    train_subset, val_subset = random_split(dataset, [n_train, n_val], generator=generator)
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader

test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


def train_one_epoch(model, loader, optimizer, criterion, epoch):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    progress_bar = tqdm(loader, desc=f'Epoch {epoch} [train]', leave=True)
    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += batch_size
        progress_bar.set_postfix({'loss': f'{total_loss/total:.4f}', 'acc': f'{correct/total:.4f}'})
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, epoch, split_name='val'):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    progress_bar = tqdm(loader, desc=f'Epoch {epoch} [{split_name}]', leave=True)
    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += batch_size
        progress_bar.set_postfix({'loss': f'{total_loss/total:.4f}', 'acc': f'{correct/total:.4f}'})
    return total_loss / total, correct / total

## 2. Hyperparameter tuning (10% data, short runs)
Same grid as Arjot's ViT tuning, so results are directly comparable: `learning_rates = [5e-5, 1e-4, 5e-4, 1e-2]`, `weight_decays = [1e-4, 1e-2]`.

In [ ]:
learning_rates = [5e-5, 1e-4, 5e-4, 1e-2]
weight_decays = [1e-4, 1e-2]

# If mlop.init() fails because the dashboard is down, set this to True to skip
# mlop logging and just track results in the local `tuning_results` list instead.
# You can always re-log to mlop later once it's back, since best_val_acc is preserved.
LOCAL_FALLBACK = False

def run_resnet18_tuning_trial(lr, wd, epochs=3):
    run_name = f'resnet18_tune_lr{lr}_wd{wd}'

    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, NUM_CLASSES)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    train_loader, val_loader = make_train_val_loaders(train_10)

    config = {'model': 'resnet18', 'lr': lr, 'weight_decay': wd, 'phase': 'tuning', 'data_fraction': '10'}

    op = None
    if not LOCAL_FALLBACK:
        try:
            op = mlop.init(name=run_name, project='food-rescue-cv', config=config)
        except Exception as e:
            print(f'mlop.init failed ({e}), continuing without live logging for this trial')

    best_val_acc = 0.0
    try:
        for epoch in range(1, epochs + 1):
            train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, epoch)
            val_loss, val_acc = evaluate(model, val_loader, criterion, epoch, split_name='val')
            if op is not None:
                try:
                    op.log({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                             'val_loss': val_loss, 'val_acc': val_acc})
                except Exception as e:
                    print(f'mlop.log failed ({e}), continuing')
            best_val_acc = max(best_val_acc, val_acc)
    finally:
        if op is not None:
            try:
                op.finish()
            except Exception:
                pass

    return {'lr': lr, 'weight_decay': wd, 'best_val_acc': best_val_acc}


tuning_results = []
for lr in learning_rates:
    for wd in weight_decays:
        print(f'--- Tuning lr={lr}, wd={wd} ---')
        result = run_resnet18_tuning_trial(lr, wd)
        tuning_results.append(result)

tuning_results.sort(key=lambda r: r['best_val_acc'], reverse=True)
print('Best config:', tuning_results[0])

## 3. Full experiment: best config across 10% / 50% / 100% data
Once the tuning loop above finishes, `tuning_results[0]` holds the best `lr`/`weight_decay`. This cell picks that automatically.

In [ ]:
BEST_LR = tuning_results[0]['lr']
BEST_WD = tuning_results[0]['weight_decay']
print(f'Using best config: lr={BEST_LR}, weight_decay={BEST_WD}')

def run_resnet18_experiment(train_dataset, fraction_label, epochs=10, aug_label='aug_on',
                             lr=BEST_LR, wd=BEST_WD):
    run_name = f'resnet18_frac{fraction_label}_{aug_label}'
    checkpoint_dir = f'{PROJECT_DIR}/checkpoints/{run_name}'
    os.makedirs(checkpoint_dir, exist_ok=True)

    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, NUM_CLASSES)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    train_loader, val_loader = make_train_val_loaders(train_dataset)

    config = {
        'model': 'resnet18', 'pretrained': True, 'data_fraction': fraction_label,
        'augmentation': aug_label, 'learning_rate': lr, 'weight_decay': wd,
        'epochs': epochs, 'batch_size': BATCH_SIZE, 'seed': SEED,
    }

    op = None
    if not LOCAL_FALLBACK:
        try:
            op = mlop.init(name=run_name, project='food-rescue-cv', config=config)
        except Exception as e:
            print(f'mlop.init failed ({e}), continuing without live logging for this run')

    best_val_acc = 0.0
    test_acc = None

    try:
        for epoch in range(1, epochs + 1):
            train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, epoch)
            val_loss, val_acc = evaluate(model, val_loader, criterion, epoch, split_name='val')

            if op is not None:
                try:
                    op.log({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                             'val_loss': val_loss, 'val_acc': val_acc})
                except Exception as e:
                    print(f'mlop.log failed ({e}), continuing')

            print(f'Epoch {epoch:02d}/{epochs} | train loss {train_loss:.4f} acc {train_acc:.4f} '
                  f'| val loss {val_loss:.4f} acc {val_acc:.4f}')

            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(), 'val_acc': val_acc,
                        'config': config}, f'{checkpoint_dir}/latest.pt')

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                            'val_acc': val_acc, 'config': config}, f'{checkpoint_dir}/best.pt')

        test_loss, test_acc = evaluate(model, test_loader, criterion, epochs, split_name='test')
        if op is not None:
            try:
                op.log({'test_loss': test_loss, 'test_acc': test_acc})
            except Exception as e:
                print(f'mlop.log failed ({e}), continuing')
        print(f'Final test | loss {test_loss:.4f} acc {test_acc:.4f}')

    finally:
        if op is not None:
            try:
                op.finish()
            except Exception:
                pass

    return model, {'best_val_acc': best_val_acc, 'test_acc': test_acc}

In [ ]:
# Smoke test on 10% data, 1 epoch, before committing to the full 10-epoch runs
_ = run_resnet18_experiment(train_10, fraction_label='10', epochs=1)

In [ ]:
# Full sweep across all three data fractions with the tuned config
EPOCHS = 10

sweep_results = {}
for fraction_label, dataset in [('10', train_10), ('50', train_50), ('100', train_100)]:
    print(f'--- Running ResNet18 on {fraction_label}% data ---')
    model, metrics = run_resnet18_experiment(dataset, fraction_label=fraction_label, epochs=EPOCHS)
    sweep_results[fraction_label] = metrics

print(sweep_results)

## Where this leaves you
At this point, if mlop.ai and Colab both cooperate, you'll have:
- Tuned ResNet18 hyperparameters (matching Arjot's grid, so comparable)
- Full 10%/50%/100% data-fraction sweep results, with checkpoints saved to Drive

**This is the point where you need Arjot in tandem**, since the next steps require both of your results together:
- Merge his ViT-B/16 sweep results with yours into one comparison table/plots
- Jointly pick the single best-performing model/fraction combo for the augmentation on/off ablation
- If mlop.ai stayed down during your runs, `sweep_results` and `tuning_results` are still in this notebook's memory. Save them to a local file so nothing is lost:
```python
import json
with open(f'{PROJECT_DIR}/results/resnet18_results.json', 'w') as f:
    json.dump({'tuning': tuning_results, 'sweep': sweep_results}, f, indent=2)
```